# Supply Chain & Inventory Intelligence Analytics

### Identifying Inventory Risks, Product Profitability, Supplier Performance, Customer Behavior and Delivery Inefficiencies

**Portfolio-grade Data Analyst Project | Google Colab**

**Workflow:** Business Problem → Objectives → Data Collection → Data Understanding → Cleaning → Feature Engineering → EDA → Statistical Analysis → Visualization → Insights → Recommendations → Conclusion → Limitations & Next Steps

> All findings must be calculated from the supplied CSV files. Do not hard-code business conclusions.

## 1. Business Problem

The company is an FMCG wholesale/distribution business supplying retailers across Delhi-NCR.

Management wants to improve:
- Inventory efficiency
- Product profitability
- Supplier reliability
- Customer profitability
- Delivery reliability

Potential problems include stockouts, slow-moving inventory, overstocking, supplier delays, customer delivery delays and discount pressure.

### Core Business Question

> **How can the company improve inventory efficiency, profitability, supplier performance and delivery reliability using historical sales, inventory, purchase and delivery data?**

## 2. Define Objectives & Business Questions

### Objectives
1. Analyze revenue and profitability trends.
2. Identify high-performing and low-performing products.
3. Identify high-value and low-value inventory.
4. Analyze inventory turnover and stockout risk.
5. Perform ABC inventory classification.
6. Identify slow-moving products.
7. Evaluate supplier performance and lead times.
8. Evaluate customer profitability and behavior.
9. Analyze delivery reliability and cost.
10. Investigate discount percentage versus margin.
11. Validate data quality before making business claims.
12. Produce actionable recommendations.

### Key Questions
**Sales:** What are revenue, profit, margin, AOV and growth trends? Which categories/products drive performance?

**Inventory:** What is the latest inventory position? Which SKUs have low turnover, high stockouts or high inventory value?

**Supplier:** Which suppliers have the highest purchase value, strongest on-time performance and longest delays?

**Customer:** Which customers generate the most revenue/profit? Which segments are most valuable?

**Delivery:** What is the on-time rate? Does distance relate to cost or delay? Which warehouses/areas have weaker service?

**Discount:** Is discount percentage associated with profit margin?

## 3. Data Collection

The supplied relational dataset contains:

| Table | Type | Purpose |
|---|---|---|
| dim_product | Dimension | Product attributes, price and reorder levels |
| dim_customer | Dimension | Retailer/customer attributes |
| dim_supplier | Dimension | Supplier attributes |
| dim_warehouse | Dimension | Distribution centers |
| dim_date | Dimension | Calendar |
| fact_sales | Fact | Sales transactions |
| fact_inventory | Fact | Periodic inventory snapshots |
| fact_purchase | Fact | Purchases and supplier delivery |
| fact_delivery | Fact | Customer delivery performance |
| README_data_dictionary | Reference | Table descriptions and keys |

The notebook supports Google Colab upload and dynamically verifies row counts/date ranges.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from scipy import stats
from IPython.display import display, Markdown
import os, warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

required = [
    "dim_product.csv","dim_customer.csv","dim_supplier.csv","dim_warehouse.csv",
    "dim_date.csv","fact_sales.csv","fact_inventory.csv","fact_purchase.csv",
    "fact_delivery.csv","README_data_dictionary.csv"
]

missing = [f for f in required if not os.path.exists(f)]
if missing:
    from google.colab import files
    print("Upload the missing files:")
    print(*missing, sep="\n")
    uploaded = files.upload()
else:
    print("All required files are already available.")

Upload the missing files:
dim_product.csv
dim_customer.csv
dim_supplier.csv
dim_warehouse.csv
dim_date.csv
fact_sales.csv
fact_inventory.csv
fact_purchase.csv
fact_delivery.csv
README_data_dictionary.csv


Saving dim_customer.csv to dim_customer.csv
Saving dim_date.csv to dim_date.csv
Saving dim_product.csv to dim_product.csv
Saving dim_supplier.csv to dim_supplier.csv
Saving dim_warehouse.csv to dim_warehouse.csv
Saving fact_delivery.csv to fact_delivery.csv
Saving fact_inventory.csv to fact_inventory.csv
Saving fact_purchase.csv to fact_purchase.csv
Saving fact_sales.csv to fact_sales.csv
Saving README_data_dictionary.csv to README_data_dictionary.csv


In [ ]:
products = pd.read_csv("dim_product.csv")
customers = pd.read_csv("dim_customer.csv")
suppliers = pd.read_csv("dim_supplier.csv")
warehouses = pd.read_csv("dim_warehouse.csv")
dates = pd.read_csv("dim_date.csv")
sales = pd.read_csv("fact_sales.csv")
inventory = pd.read_csv("fact_inventory.csv")
purchases = pd.read_csv("fact_purchase.csv")
deliveries = pd.read_csv("fact_delivery.csv")
data_dictionary = pd.read_csv("README_data_dictionary.csv")

# Date parsing
for c in ["date"]:
    dates[c] = pd.to_datetime(dates[c], errors="coerce")
sales["order_date"] = pd.to_datetime(sales["order_date"], errors="coerce")
inventory["date"] = pd.to_datetime(inventory["date"], errors="coerce")
for c in ["purchase_date","expected_delivery_date","actual_delivery_date"]:
    purchases[c] = pd.to_datetime(purchases[c], errors="coerce")
deliveries["delivery_date"] = pd.to_datetime(deliveries["delivery_date"], errors="coerce")

tables = {
    "dim_product": products, "dim_customer": customers, "dim_supplier": suppliers,
    "dim_warehouse": warehouses, "dim_date": dates, "fact_sales": sales,
    "fact_inventory": inventory, "fact_purchase": purchases, "fact_delivery": deliveries
}
print("Loaded", len(tables), "tables.")

Loaded 9 tables.


# 4. Data Understanding / Exploration

First inspect structure, data types, missing values, duplicates, unique values, numerical summaries and date coverage.

In [ ]:
def profile_dataframe(df, name):
    return {
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
        "numeric_columns": len(df.select_dtypes(include=np.number).columns),
        "date_columns": len([c for c in df.columns if np.issubdtype(df[c].dtype, np.datetime64)])
    }

profile = pd.DataFrame([profile_dataframe(df,n) for n,df in tables.items()])
display(profile)

for name, df in tables.items():
    print(f"\n{name}: {df.shape}")
    display(df.head(3))

,table,rows,columns,duplicate_rows,missing_cells,numeric_columns,date_columns
0,dim_product,30,9,0,0,4,0
1,dim_customer,100,8,0,0,2,0
2,dim_supplier,15,5,0,0,2,0
3,dim_warehouse,4,5,0,0,1,0
4,dim_date,365,9,0,0,4,1
5,fact_sales,3000,12,0,0,6,1
6,fact_inventory,1440,10,0,0,7,1
7,fact_purchase,1200,10,0,0,3,3
8,fact_delivery,2200,10,0,0,4,1



dim_product: (30, 9)


,product_id,product_name,category,sub_category,brand,unit_cost,selling_price,reorder_level,shelf_life_days
0,P001,Fortune Mustard Oil 1L,Edible Oil,Mustard Oil,Fortune,118,145,120,365
1,P002,Fortune Sunflower Oil 1L,Edible Oil,Sunflower Oil,Fortune,112,139,120,365
2,P003,Tata Salt 1kg,Staples,Salt,Tata,20,28,250,730



dim_customer: (100, 8)


,customer_id,customer_name,city,area,customer_type,customer_segment,credit_limit,payment_terms_days
0,C001,Shahdara General Store 001,Noida,Shahdara,Kirana,C,150000,30
1,C002,Dilshad Garden General Store 002,Delhi,Dilshad Garden,Kirana,C,100000,30
2,C003,Vasundhara Enclave General Store 003,Delhi,Vasundhara Enclave,Mini Mart,C,50000,7


...

,date,warehouse_id,product_id,opening_stock,received_quantity,sold_quantity,closing_stock,damaged_quantity,stockout_flag,inventory_value
0,2025-01-01,W001,P001,307,152,65,390,4,0,46020
1,2025-01-01,W001,P002,237,153,44,345,1,0,38640
2,2025-01-01,W001,P003,454,110,23,539,2,0,10780



fact_purchase: (1200, 10)


,purchase_id,purchase_date,supplier_id,product_id,warehouse_id,quantity,unit_cost,purchase_value,expected_delivery_date,actual_delivery_date
0,PO00898,2025-01-01,S003,P021,W003,100,112,11200,2025-01-07,2025-01-07
1,PO00900,2025-01-01,S015,P027,W002,100,135,13500,2025-01-07,2025-01-07
2,PO00078,2025-01-02,S007,P014,W004,200,62,12400,2025-01-04,2025-01-04



fact_delivery: (2200, 10)


,delivery_id,order_id,delivery_date,customer_id,warehouse_id,distance_km,delivery_cost,expected_days,actual_days,delivery_status
0,D00001,O02736,2025-08-09,C084,W003,30.30,211.98,2,2,On Time
1,D00002,O00441,2025-05-27,C099,W001,9.40,124.95,3,3,On Time
2,D00003,O00516,2025-08-12,C017,W001,13.90,131.10,4,2,On Time


In [ ]:
# Independent row-count verification
expected = {
    "dim_product":30, "dim_customer":100, "dim_supplier":15, "dim_warehouse":4,
    "dim_date":365, "fact_sales":3000, "fact_inventory":1440,
    "fact_purchase":1200, "fact_delivery":2200
}
verification = pd.DataFrame({
    "table": expected.keys(),
    "expected_rows": expected.values(),
    "actual_rows": [len(tables[k]) for k in expected]
})
verification["match"] = verification.expected_rows.eq(verification.actual_rows)
display(verification)

# Date coverage
for name, cols in {
    "dim_date":["date"], "fact_sales":["order_date"], "fact_inventory":["date"],
    "fact_purchase":["purchase_date","expected_delivery_date","actual_delivery_date"],
    "fact_delivery":["delivery_date"]
}.items():
    for c in cols:
        s=tables[name][c]
        print(f"{name}.{c}: {s.min().date()} -> {s.max().date()} | invalid: {s.isna().sum()}")

,table,expected_rows,actual_rows,match
0,dim_product,30,30,True
1,dim_customer,100,100,True
2,dim_supplier,15,15,True
3,dim_warehouse,4,4,True
4,dim_date,365,365,True
5,fact_sales,3000,3000,True
6,fact_inventory,1440,1440,True
7,fact_purchase,1200,1200,True
8,fact_delivery,2200,2200,True


dim_date.date: 2025-01-01 -> 2025-12-31 | invalid: 0
fact_sales.order_date: 2025-01-01 -> 2025-12-31 | invalid: 0
fact_inventory.date: 2025-01-01 -> 2025-12-01 | invalid: 0
fact_purchase.purchase_date: 2025-01-01 -> 2025-12-31 | invalid: 0
fact_purchase.expected_delivery_date: 2025-01-04 -> 2026-01-08 | invalid: 0
fact_purchase.actual_delivery_date: 2025-01-04 -> 2026-01-08 | invalid: 0
fact_delivery.delivery_date: 2025-01-02 -> 2026-01-05 | invalid: 0


## Data Relationship Validation

Validate all fact-to-dimension foreign keys and primary-key uniqueness. Never assume the relational model is valid.

In [ ]:
key_specs = {
    "dim_product":"product_id","dim_customer":"customer_id","dim_supplier":"supplier_id",
    "dim_warehouse":"warehouse_id","fact_sales":"order_id",
    "fact_purchase":"purchase_id","fact_delivery":"delivery_id"
}
key_report = pd.DataFrame([{
    "table":t, "key":k, "null_keys":int(tables[t][k].isna().sum()),
    "duplicate_keys":int(tables[t][k].duplicated().sum()),
    "unique_keys":int(tables[t][k].nunique())
} for t,k in key_specs.items()])
display(key_report)

relations = [
("fact_sales","product_id","dim_product","product_id"),
("fact_sales","customer_id","dim_customer","customer_id"),
("fact_sales","warehouse_id","dim_warehouse","warehouse_id"),
("fact_inventory","product_id","dim_product","product_id"),
("fact_inventory","warehouse_id","dim_warehouse","warehouse_id"),
("fact_purchase","product_id","dim_product","product_id"),
("fact_purchase","supplier_id","dim_supplier","supplier_id"),
("fact_purchase","warehouse_id","dim_warehouse","warehouse_id"),
("fact_delivery","order_id","fact_sales","order_id"),
("fact_delivery","customer_id","dim_customer","customer_id"),
("fact_delivery","warehouse_id","dim_warehouse","warehouse_id")
]
relationship_report = pd.DataFrame([{
    "fact_table":ft, "foreign_key":fk, "parent_table":dt,
    "orphan_rows":int((~tables[ft][fk].isin(tables[dt][dk])).sum())
} for ft,fk,dt,dk in relations])
relationship_report["valid"] = relationship_report.orphan_rows.eq(0)
display(relationship_report)

,table,key,null_keys,duplicate_keys,unique_keys
0,dim_product,product_id,0,0,30
1,dim_customer,customer_id,0,0,100
2,dim_supplier,supplier_id,0,0,15
3,dim_warehouse,warehouse_id,0,0,4
4,fact_sales,order_id,0,0,3000
5,fact_purchase,purchase_id,0,0,1200
6,fact_delivery,delivery_id,0,0,2200


,fact_table,foreign_key,parent_table,orphan_rows,valid
0,fact_sales,product_id,dim_product,0,True
1,fact_sales,customer_id,dim_customer,0,True
2,fact_sales,warehouse_id,dim_warehouse,0,True
3,fact_inventory,product_id,dim_product,0,True
4,fact_inventory,warehouse_id,dim_warehouse,0,True
5,fact_purchase,product_id,dim_product,0,True
6,fact_purchase,supplier_id,dim_supplier,0,True
7,fact_purchase,warehouse_id,dim_warehouse,0,True
8,fact_delivery,order_id,fact_sales,0,True
9,fact_delivery,customer_id,dim_customer,0,True


## Data Quality Checks

Check:
- Missing values
- Duplicate records
- Negative values
- Invalid dates
- Sales/profit calculation consistency
- Inventory stock equation
- Inventory-sales reconciliation

Known inconsistencies must be **reported, not silently corrected**.

In [ ]:
# Missing values and duplicates
quality = pd.DataFrame([{
    "table":n,
    "missing_cells":int(df.isna().sum().sum()),
    "duplicate_rows":int(df.duplicated().sum())
} for n,df in tables.items()])
display(quality)

# Negative values
non_negative = {
    "fact_sales":["quantity","unit_price","discount","sales_amount","cost_amount","profit"],
    "fact_inventory":["opening_stock","received_quantity","sold_quantity","closing_stock","damaged_quantity","inventory_value"],
    "fact_purchase":["quantity","unit_cost","purchase_value"],
    "fact_delivery":["distance_km","delivery_cost","expected_days","actual_days"]
}
negative = []
for t, cols in non_negative.items():
    for c in cols:
        negative.append({"table":t,"column":c,"negative_rows":int((tables[t][c]<0).sum())})
negative = pd.DataFrame(negative)
display(negative[negative.negative_rows.gt(0)])

,table,missing_cells,duplicate_rows
0,dim_product,0,0
1,dim_customer,0,0
2,dim_supplier,0,0
3,dim_warehouse,0,0
4,dim_date,0,0
5,fact_sales,0,0
6,fact_inventory,0,0
7,fact_purchase,0,0
8,fact_delivery,0,0


,table,column,negative_rows


In [ ]:
# Logical sales calculations
expected_sales = sales.quantity * sales.unit_price - sales.discount
expected_profit = sales.sales_amount - sales.cost_amount

print("Sales amount mismatches:", int((~np.isclose(sales.sales_amount, expected_sales)).sum()))
print("Profit mismatches:", int((~np.isclose(sales.profit, expected_profit)).sum()))

# Date logic
purchase_invalid = (purchases.purchase_date > purchases.expected_delivery_date).sum()
purchase_order_issue = (purchases.expected_delivery_date > purchases.actual_delivery_date).sum()

delivery_lookup = sales[["order_id","order_date"]].drop_duplicates()
delivery_check = deliveries.merge(delivery_lookup, on="order_id", how="left")
delivery_before_order = (delivery_check.delivery_date < delivery_check.order_date).sum()

print("Purchase after expected delivery:", int(purchase_invalid))
print("Expected delivery after actual delivery:", int(purchase_order_issue))
print("Delivery before order:", int(delivery_before_order))

Sales amount mismatches: 0
Profit mismatches: 0
Purchase after expected delivery: 0
Expected delivery after actual delivery: 47
Delivery before order: 0


## Dataset-Specific Inventory Validation

### 1. Inventory vs sales reconciliation

Compare `fact_inventory.sold_quantity` with `fact_sales.quantity` at **month + product + warehouse**.

### 2. Inventory stock equation

Test:

**Opening + Received − Sold − Damaged = Closing**

### 3. Inventory snapshot warning

Do **not** sum monthly inventory snapshots and call the result "current inventory." Use the latest snapshot for current/end-period inventory and the average snapshot value for turnover proxies.

In [ ]:
sales_r = sales.copy()
inv_r = inventory.copy()
sales_r["month"] = sales_r.order_date.dt.to_period("M").astype(str)
inv_r["month"] = inv_r.date.dt.to_period("M").astype(str)

s = sales_r.groupby(["month","product_id","warehouse_id"],as_index=False).quantity.sum().rename(columns={"quantity":"sales_quantity"})
i = inv_r.groupby(["month","product_id","warehouse_id"],as_index=False).sold_quantity.sum()

recon = s.merge(i,on=["month","product_id","warehouse_id"],how="outer").fillna(0)
recon["difference"] = recon.sales_quantity - recon.sold_quantity

recon_summary = pd.Series({
    "comparison_groups":len(recon),
    "exact_matches":int(np.isclose(recon.difference,0).sum()),
    "groups_with_difference":int((~np.isclose(recon.difference,0)).sum()),
    "mean_absolute_difference":recon.difference.abs().mean(),
    "sales_quantity_total":recon.sales_quantity.sum(),
    "inventory_sold_quantity_total":recon.sold_quantity.sum(),
    "total_difference":recon.difference.sum()
})
display(recon_summary.to_frame("value"))

inventory["stock_equation_difference"] = (
    inventory.opening_stock + inventory.received_quantity
    - inventory.sold_quantity - inventory.damaged_quantity
    - inventory.closing_stock
)
stock_check = pd.Series({
    "rows_checked":len(inventory),
    "mismatches":int((~np.isclose(inventory.stock_equation_difference,0)).sum()),
    "max_abs_difference":inventory.stock_equation_difference.abs().max(),
    "mean_abs_difference":inventory.stock_equation_difference.abs().mean()
})
display(stock_check.to_frame("value"))

,value
comparison_groups,"1,440.00"
exact_matches,16.00
groups_with_difference,"1,424.00"
mean_absolute_difference,53.75
sales_quantity_total,"40,934.00"
inventory_sold_quantity_total,"94,045.00"
total_difference,"-53,111.00"


,value
rows_checked,"1,440.00"
mismatches,52.00
max_abs_difference,650.00
mean_abs_difference,13.14


# 5. Data Cleaning & Preparation

Only justified transformations are applied:
1. Parse dates.
2. Validate numeric fields.
3. Validate keys.
4. Keep source measures unchanged.
5. Flag inconsistent calculations instead of overwriting them.
6. Sort time-series data where needed.

In [ ]:
cleaning_summary = pd.DataFrame([{
    "table":n,
    "original_rows":len(df),
    "duplicates":int(df.duplicated().sum()),
    "missing_cells":int(df.isna().sum().sum()),
    "final_rows":len(df)
} for n,df in tables.items()])
display(cleaning_summary)

,table,original_rows,duplicates,missing_cells,final_rows
0,dim_product,30,0,0,30
1,dim_customer,100,0,0,100
2,dim_supplier,15,0,0,15
3,dim_warehouse,4,0,0,4
4,dim_date,365,0,0,365
5,fact_sales,3000,0,0,3000
6,fact_inventory,1440,0,0,1440
7,fact_purchase,1200,0,0,1200
8,fact_delivery,2200,0,0,2200


# 6. Feature Engineering

Create business-ready measures while protecting against division by zero.

In [ ]:
# Sales features
sales["gross_sales"] = sales.quantity * sales.unit_price
sales["discount_pct"] = np.where(sales.gross_sales>0, sales.discount/sales.gross_sales*100, 0)
sales["profit_margin_pct"] = np.where(sales.sales_amount>0, sales.profit/sales.sales_amount*100, 0)
sales["profit_per_unit"] = np.where(sales.quantity>0, sales.profit/sales.quantity, 0)
sales["order_month"] = sales.order_date.dt.to_period("M").astype(str)
sales["order_month_name"] = sales.order_date.dt.strftime("%B")
sales["order_quarter"] = "Q" + sales.order_date.dt.quarter.astype(str)
sales["order_year"] = sales.order_date.dt.year

# Delivery features
deliveries["delivery_delay_days"] = deliveries.actual_days - deliveries.expected_days
deliveries["delivery_performance"] = np.select(
    [deliveries.delivery_delay_days<=0,
     deliveries.delivery_delay_days.between(1,2),
     deliveries.delivery_delay_days>=3],
    ["On Time","Slightly Delayed","Highly Delayed"], default="Unknown")
deliveries["cost_per_km"] = np.where(deliveries.distance_km>0, deliveries.delivery_cost/deliveries.distance_km, np.nan)

# Supplier features
purchases["supplier_delay_days"] = (
    purchases.actual_delivery_date - purchases.expected_delivery_date
).dt.days
purchases["supplier_on_time"] = purchases.supplier_delay_days <= 0

display(sales.head(2))

,order_id,order_date,customer_id,product_id,warehouse_id,quantity,unit_price,discount,sales_amount,cost_amount,profit,payment_method,gross_sales,discount_pct,profit_margin_pct,profit_per_unit,order_month,order_month_name,order_quarter,order_year
0,O00207,2025-01-01,C092,P009,W003,12,82,0.00,984.00,744,240.00,UPI,984,0.00,24.39,20.00,2025-01,January,Q1,2025
1,O00244,2025-01-01,C019,P002,W002,3,139,33.36,383.64,336,47.64,Credit,417,8.00,12.42,15.88,2025-01,January,Q1,2025


## Inventory Turnover

Use:

**Inventory Turnover = Annual COGS / Average Inventory Value**

The average inventory value is based on available monthly snapshots. This is an analytical proxy, not a continuous stock-ledger calculation.

In [ ]:
inventory["month"] = inventory.date.dt.to_period("M").astype(str)

inv_monthly = inventory.groupby(["month","product_id"],as_index=False).agg(
    inventory_value=("inventory_value","sum"),
    closing_stock=("closing_stock","sum"),
    stockout_observations=("stockout_flag","sum")
)
inv_avg = inv_monthly.groupby("product_id",as_index=False).agg(
    average_inventory_value=("inventory_value","mean"),
    average_closing_stock=("closing_stock","mean"),
    stockout_observations=("stockout_observations","sum")
)
cogs = sales.groupby("product_id",as_index=False).cost_amount.sum().rename(columns={"cost_amount":"annual_cogs"})

inventory_metrics = inv_avg.merge(cogs,on="product_id",how="left").fillna({"annual_cogs":0})
inventory_metrics["inventory_turnover"] = np.where(
    inventory_metrics.average_inventory_value>0,
    inventory_metrics.annual_cogs/inventory_metrics.average_inventory_value,
    np.nan
)
inventory_metrics["days_inventory"] = np.where(
    inventory_metrics.inventory_turnover>0,
    365/inventory_metrics.inventory_turnover,
    np.nan
)
inventory_metrics["stockout_risk"] = np.select(
    [inventory_metrics.stockout_observations>=4,
     inventory_metrics.stockout_observations.between(1,3),
     inventory_metrics.stockout_observations.eq(0)],
    ["High","Medium","Low"], default="Unknown"
)
inventory_metrics = inventory_metrics.merge(
    products[["product_id","product_name","category","reorder_level"]],
    on="product_id",how="left"
)
display(inventory_metrics.sort_values("inventory_turnover").head(10))

,product_id,average_inventory_value,average_closing_stock,stockout_observations,annual_cogs,inventory_turnover,days_inventory,stockout_risk,product_name,category,reorder_level
17,P018,"52,535.00","1,382.50",11,41268,0.79,464.65,High,Coca-Cola 750ml,Beverages,250
18,P019,"46,494.00","1,291.50",20,39564,0.85,428.93,High,Pepsi 750ml,Beverages,250
1,P002,"150,229.33","1,341.33",3,128016,0.85,428.33,Medium,Fortune Sunflower Oil 1L,Edible Oil,120
4,P005,"510,737.50","1,309.58",3,438360,0.86,425.27,Medium,Daawat Basmati Rice 5kg,Staples,80
20,P021,"149,249.33","1,332.58",5,134848,0.90,403.98,High,Surf Excel 1kg,Home Care,120
2,P003,"25,290.00","1,264.50",15,23160,0.92,398.57,High,Tata Salt 1kg,Staples,250
23,P024,"131,118.75","1,248.75",1,121275,0.92,394.63,Medium,Fortune Soya Health Oil 1L,Edible Oil,110
0,P001,"154,983.17","1,313.42",3,144432,0.93,391.66,Medium,Fortune Mustard Oil 1L,Edible Oil,120
6,P007,"159,630.00","1,330.25",1,150480,0.94,387.19,Medium,Bru Instant Coffee 100g,Beverages,70
9,P010,"180,213.00","1,365.25",8,182160,1.01,361.10,High,Maggi Noodles 12 Pack,Instant Food,180


## ABC Analysis

Classify products using annual revenue contribution:

- A: cumulative contribution ≤ 80%
- B: >80% and ≤95%
- C: >95%

This is a management convention used for prioritization.

In [ ]:
abc = sales.groupby("product_id",as_index=False).sales_amount.sum().rename(columns={"sales_amount":"annual_revenue"})
abc = abc.sort_values("annual_revenue",ascending=False)
abc["revenue_share_pct"] = abc.annual_revenue/abc.annual_revenue.sum()*100
abc["cumulative_share_pct"] = abc.revenue_share_pct.cumsum()
abc["abc_class"] = np.select(
    [abc.cumulative_share_pct<=80, abc.cumulative_share_pct<=95],
    ["A","B"], default="C"
)
abc = abc.merge(products[["product_id","product_name","category"]],on="product_id",how="left")
display(abc)

,product_id,annual_revenue,revenue_share_pct,cumulative_share_pct,abc_class,product_name,category
0,P023,"624,738.30",9.50,9.50,A,India Gate Rice 5kg,Staples
1,P005,"509,771.40",7.75,17.25,A,Daawat Basmati Rice 5kg,Staples
2,P029,"486,634.95",7.40,24.65,A,Amul Butter 500g,Dairy
3,P012,"414,513.40",6.30,30.95,A,Ariel Matic 2kg,Home Care
4,P011,"394,686.45",6.00,36.95,A,Surf Excel 2kg,Home Care
5,P006,"337,422.80",5.13,42.09,A,Tata Tea 500g,Beverages
6,P004,"328,282.85",4.99,47.08,A,Aashirvaad Atta 5kg,Staples
7,P028,"284,088.00",4.32,51.40,A,London Dairy Ice Cream 1L,Frozen
8,P015,"271,224.00",4.12,55.52,A,Dove Soap 4 Pack,Personal Care
9,P027,"252,920.90",3.85,59.37,A,Pears Soap 4 Pack,Personal Care


## Customer RFM-Style Segmentation

Calculate Recency, Frequency and Monetary Value. Use a transparent heuristic segmentation rather than presenting it as a validated predictive model.

In [ ]:
analysis_date = sales.order_date.max()
rfm = sales.groupby("customer_id").agg(
    recency_days=("order_date",lambda x:(analysis_date-x.max()).days),
    frequency=("order_id","nunique"),
    monetary=("sales_amount","sum")
).reset_index()

rfm["R_score"] = pd.qcut(rfm.recency_days.rank(method="first",ascending=True),4,labels=[4,3,2,1]).astype(int)
rfm["F_score"] = pd.qcut(rfm.frequency.rank(method="first"),4,labels=[1,2,3,4]).astype(int)
rfm["M_score"] = pd.qcut(rfm.monetary.rank(method="first"),4,labels=[1,2,3,4]).astype(int)
rfm["RFM_score"] = rfm.R_score + rfm.F_score + rfm.M_score
rfm["segment"] = np.select(
    [rfm.RFM_score>=10,(rfm.RFM_score>=8)&(rfm.RFM_score<10),
     (rfm.RFM_score>=6)&(rfm.RFM_score<8),rfm.R_score<=2],
    ["High Value","Loyal / Regular","Regular","At Risk"],default="Low Value"
)
rfm = rfm.merge(customers[["customer_id","customer_name","city","area","customer_type","customer_segment"]],
                on="customer_id",how="left")
display(rfm.head(10))

,customer_id,recency_days,frequency,monetary,R_score,F_score,M_score,RFM_score,segment,customer_name,city,area,customer_type,customer_segment
0,C001,5,29,"52,337.35",3,2,2,7,Regular,Shahdara General Store 001,Noida,Shahdara,Kirana,C
1,C002,2,40,"77,093.82",4,4,3,11,High Value,Dilshad Garden General Store 002,Delhi,Dilshad Garden,Kirana,C
2,C003,1,31,"103,604.28",4,3,4,11,High Value,Vasundhara Enclave General Store 003,Delhi,Vasundhara Enclave,Mini Mart,C
3,C004,0,28,"41,020.45",4,2,1,7,Regular,Gazipur General Store 004,Delhi,Gazipur,Kirana,B
4,C005,5,22,"41,644.82",3,1,1,5,Low Value,Dilshad Garden General Store 005,Delhi,Dilshad Garden,Convenience Store,B
5,C006,1,29,"55,932.86",4,2,2,8,Loyal / Regular,Laxmi Nagar General Store 006,Delhi,Laxmi Nagar,Kirana,C
6,C007,4,26,"49,402.40",3,1,1,5,Low Value,Shahdara General Store 007,Delhi,Shahdara,Convenience Store,C
7,C008,31,18,"24,244.72",1,1,1,3,At Risk,Dilshad Garden General Store 008,Delhi,Dilshad Garden,Supermarket,B
8,C009,22,37,"106,413.15",1,4,4,9,Loyal / Regular,Kondli General Store 009,Delhi,Kondli,Convenience Store,B
9,C010,23,26,"40,695.48",1,1,1,3,At Risk,Vasundhara Enclave General Store 010,Delhi,Vasundhara Enclave,Supermarket,A


# 7. Exploratory Data Analysis (EDA)

## Executive KPIs

In [ ]:
total_revenue = sales.sales_amount.sum()
total_profit = sales.profit.sum()
profit_margin = total_profit/total_revenue*100
total_orders = sales.order_id.nunique()
total_units = sales.quantity.sum()
aov = total_revenue/total_orders

latest_inventory_date = inventory.date.max()
latest_inventory = inventory.loc[inventory.date.eq(latest_inventory_date),"inventory_value"].sum()
stockout_rate = inventory.stockout_flag.mean()*100
supplier_on_time_rate = purchases.supplier_on_time.mean()*100
delivery_on_time_rate = (deliveries.delivery_delay_days<=0).mean()*100

kpi = pd.DataFrame({
    "KPI":["Total Revenue","Total Profit","Profit Margin","Orders","Units Sold",
           "Average Order Value","Latest Inventory Value","Stockout Rate",
           "Supplier On-Time Rate","Delivery On-Time Rate"],
    "Value":[total_revenue,total_profit,profit_margin,total_orders,total_units,
             aov,latest_inventory,stockout_rate,supplier_on_time_rate,delivery_on_time_rate]
})
display(kpi)

,KPI,Value
0,Total Revenue,"6,576,610.86"
1,Total Profit,"1,071,241.86"
2,Profit Margin,16.29
3,Orders,"3,000.00"
4,Units Sold,"40,934.00"
5,Average Order Value,"2,192.20"
6,Latest Inventory Value,"5,117,961.00"
7,Stockout Rate,14.65
8,Supplier On-Time Rate,74.00
9,Delivery On-Time Rate,46.50


## A. Sales Trend

**Question:** How do revenue, profit and margin change over time?

In [ ]:
monthly = sales.groupby("order_month",as_index=False).agg(
    revenue=("sales_amount","sum"), profit=("profit","sum"),
    units=("quantity","sum"), orders=("order_id","nunique")
)
monthly["profit_margin_pct"] = monthly.profit/monthly.revenue*100
monthly["mom_growth_pct"] = monthly.revenue.pct_change()*100

px.line(monthly,x="order_month",y="revenue",markers=True,
        title="Monthly Revenue Trend",labels={"order_month":"Month","revenue":"Revenue (₹)"}).show()
px.line(monthly,x="order_month",y="profit",markers=True,
        title="Monthly Profit Trend",labels={"order_month":"Month","profit":"Profit (₹)"}).show()
px.line(monthly,x="order_month",y="profit_margin_pct",markers=True,
        title="Monthly Profit Margin Trend",labels={"order_month":"Month","profit_margin_pct":"Margin (%)"}).show()

## B. Category & Product Analysis

In [ ]:
category = sales.merge(products[["product_id","category"]],on="product_id").groupby("category",as_index=False).agg(
    revenue=("sales_amount","sum"),profit=("profit","sum"),units=("quantity","sum")
)
category["margin_pct"] = category.profit/category.revenue*100
display(category.sort_values("revenue",ascending=False))

px.bar(category.sort_values("revenue"),x="revenue",y="category",orientation="h",
       color="category",text_auto=".2s",title="Revenue by Category",
       labels={"revenue":"Revenue (₹)"}).show()
px.bar(category.sort_values("profit"),x="profit",y="category",orientation="h",
       color="category",text_auto=".2s",title="Profit by Category",
       labels={"profit":"Profit (₹)"}).show()

,category,revenue,profit,units,margin_pct
9,Staples,"1,692,074.38","243,564.38",6707,14.39
5,Home Care,"1,423,045.86","234,881.86",8310,16.51
7,Personal Care,"807,296.94","146,268.94",5998,18.12
0,Beverages,"699,393.14","126,057.14",6311,18.02
1,Dairy,"573,177.65","65,881.65",3176,11.49
2,Edible Oil,"469,567.97","75,844.97",3522,16.15
3,Frozen,"284,088.00","60,348.00",1243,21.24
6,Instant Food,"223,473.60","41,313.60",1380,18.49
4,Grocery,"203,801.40","39,791.40",1562,19.52
8,Snacks,"200,691.92","37,289.92",2725,18.58


In [ ]:
product_summary = sales.merge(products[["product_id","product_name","category"]],on="product_id").groupby(
    ["product_id","product_name","category"],as_index=False
).agg(revenue=("sales_amount","sum"),profit=("profit","sum"),units=("quantity","sum"),discount=("discount","sum"))
product_summary["margin_pct"] = product_summary.profit/product_summary.revenue*100

top_revenue = product_summary.nlargest(10,"revenue")
top_profit = product_summary.nlargest(10,"profit")

px.bar(top_revenue.sort_values("revenue"),x="revenue",y="product_name",orientation="h",
       color="category",text_auto=".2s",title="Top 10 Products by Revenue").show()
px.bar(top_profit.sort_values("profit"),x="profit",y="product_name",orientation="h",
       color="category",text_auto=".2s",title="Top 10 Products by Profit").show()
px.scatter(product_summary,x="revenue",y="profit",size="units",color="category",
           hover_name="product_name",hover_data=["margin_pct"],
           title="Product Revenue vs Profitability").show()

## C. Discount vs Profit Margin

In [ ]:
discount_plot = sales.merge(products[["product_id","category","product_name"]],on="product_id")
px.scatter(discount_plot,x="discount_pct",y="profit_margin_pct",size="sales_amount",
           color="category",hover_name="product_name",opacity=.65,
           title="Discount Percentage vs Profit Margin",
           labels={"discount_pct":"Discount (%)","profit_margin_pct":"Profit Margin (%)"}).show()

## D. Inventory Analysis

In [ ]:
latest = inventory[inventory.date.eq(latest_inventory_date)].groupby("product_id",as_index=False).agg(
    inventory_value=("inventory_value","sum"),closing_stock=("closing_stock","sum"),
    stockout_observations=("stockout_flag","sum")
).merge(products[["product_id","product_name","category"]],on="product_id",how="left")

px.bar(latest.nlargest(10,"inventory_value").sort_values("inventory_value"),
       x="inventory_value",y="product_name",orientation="h",color="category",
       text_auto=".2s",title="Top Products by Latest Inventory Value").show()

px.bar(inventory_metrics.nsmallest(10,"inventory_turnover").sort_values("inventory_turnover"),
       x="inventory_turnover",y="product_name",orientation="h",color="stockout_risk",
       text_auto=".2f",title="Lowest Inventory Turnover Products").show()

stockouts = inventory.groupby("product_id",as_index=False).stockout_flag.mean()
stockouts["stockout_rate_pct"] = stockouts.stockout_flag*100
stockouts = stockouts.merge(products[["product_id","product_name","category"]],on="product_id")
px.bar(stockouts.nlargest(10,"stockout_rate_pct").sort_values("stockout_rate_pct"),
       x="stockout_rate_pct",y="product_name",orientation="h",color="category",
       text_auto=".1f",title="Highest Product Stockout Rates").show()

inv_trend = inventory.groupby("month",as_index=False).inventory_value.sum()
px.line(inv_trend,x="month",y="inventory_value",markers=True,
        title="Monthly Inventory Snapshot Value",
        labels={"inventory_value":"Inventory Value (₹)"}).show()

## E. ABC Analysis

In [ ]:
abc_counts = abc.abc_class.value_counts().reindex(["A","B","C"]).fillna(0).reset_index()
abc_counts.columns=["abc_class","product_count"]
px.bar(abc_counts,x="abc_class",y="product_count",text="product_count",
       title="Number of Products by ABC Class").show()
display(abc)

,product_id,annual_revenue,revenue_share_pct,cumulative_share_pct,abc_class,product_name,category
0,P023,"624,738.30",9.50,9.50,A,India Gate Rice 5kg,Staples
1,P005,"509,771.40",7.75,17.25,A,Daawat Basmati Rice 5kg,Staples
2,P029,"486,634.95",7.40,24.65,A,Amul Butter 500g,Dairy
3,P012,"414,513.40",6.30,30.95,A,Ariel Matic 2kg,Home Care
4,P011,"394,686.45",6.00,36.95,A,Surf Excel 2kg,Home Care
5,P006,"337,422.80",5.13,42.09,A,Tata Tea 500g,Beverages
6,P004,"328,282.85",4.99,47.08,A,Aashirvaad Atta 5kg,Staples
7,P028,"284,088.00",4.32,51.40,A,London Dairy Ice Cream 1L,Frozen
8,P015,"271,224.00",4.12,55.52,A,Dove Soap 4 Pack,Personal Care
9,P027,"252,920.90",3.85,59.37,A,Pears Soap 4 Pack,Personal Care


## F. Supplier Performance

In [ ]:
supplier_summary = purchases.groupby("supplier_id",as_index=False).agg(
    purchase_orders=("purchase_id","nunique"),
    purchase_volume=("quantity","sum"),
    purchase_value=("purchase_value","sum"),
    avg_delay_days=("supplier_delay_days","mean"),
    on_time_rate=("supplier_on_time","mean")
)
supplier_summary["on_time_rate_pct"] = supplier_summary.on_time_rate*100
supplier_summary = supplier_summary.merge(
    suppliers[["supplier_id","supplier_name","city","lead_time_days","supplier_rating"]],
    on="supplier_id",how="left"
)

display(supplier_summary.sort_values("purchase_value",ascending=False))
px.bar(supplier_summary.sort_values("on_time_rate_pct"),x="on_time_rate_pct",y="supplier_name",
       orientation="h",color="supplier_rating",text_auto=".1f",
       title="Supplier On-Time Delivery Rate").show()
px.bar(supplier_summary.sort_values("avg_delay_days"),x="avg_delay_days",y="supplier_name",
       orientation="h",color="supplier_rating",text_auto=".2f",
       title="Average Supplier Delivery Delay").show()

,supplier_id,purchase_orders,purchase_volume,purchase_value,avg_delay_days,on_time_rate,on_time_rate_pct,supplier_name,city,lead_time_days,supplier_rating
14,S015,92,16450,2268900,0.48,0.75,75.00,Galaxy Supply Network,Delhi,6,4.20
6,S007,88,16460,2256910,0.50,0.74,73.86,Prime FMCG Traders,Delhi,2,4.40
5,S006,70,15540,2241140,0.16,0.84,84.29,Capital Distributors,Delhi,6,3.50
4,S005,82,16810,2190250,0.50,0.74,74.39,Bharat Wholesale Hub,Delhi,7,3.60
10,S011,89,15860,2182580,0.69,0.67,67.42,Apex Consumer Goods,Ghaziabad,5,4.50
1,S002,81,14680,2165040,0.48,0.75,75.31,Metro FMCG Supply,Noida,6,4.10
7,S008,82,14070,2086240,0.70,0.68,68.29,SRS Supply Co.,Ghaziabad,5,3.60
12,S013,81,14920,2071650,0.49,0.77,76.54,Om Sai Wholesale,Noida,2,4.10
3,S004,81,13180,2018180,0.43,0.75,75.31,North India Foods,Noida,2,3.60
9,S010,78,15390,1860140,0.49,0.73,73.08,East Delhi Distributors,Ghaziabad,7,4.50


## G. Delivery Performance

In [ ]:
status = pd.DataFrame({"status":np.where(deliveries.delivery_delay_days<=0,"On Time","Delayed")})
status = status.value_counts().reset_index()
status.columns=["status","deliveries"]
px.pie(status,names="status",values="deliveries",hole=.45,title="Customer Delivery Status").show()

px.scatter(deliveries,x="distance_km",y="delivery_cost",color="delivery_performance",
           hover_data=["warehouse_id","customer_id","delivery_delay_days"],
           title="Delivery Distance vs Delivery Cost").show()

px.scatter(deliveries,x="distance_km",y="delivery_delay_days",color="delivery_performance",
           hover_data=["warehouse_id","customer_id","delivery_cost"],
           title="Delivery Distance vs Delivery Delay").show()

## H. Customer Analysis

In [ ]:
customer_summary = sales.groupby("customer_id",as_index=False).agg(
    revenue=("sales_amount","sum"),profit=("profit","sum"),
    orders=("order_id","nunique"),units=("quantity","sum")
).merge(customers[["customer_id","customer_name","city","area","customer_type","customer_segment"]],
        on="customer_id",how="left")
customer_summary["margin_pct"] = customer_summary.profit/customer_summary.revenue*100
customer_summary["aov"] = customer_summary.revenue/customer_summary.orders

px.scatter(customer_summary,x="revenue",y="profit",size="orders",color="customer_segment",
           hover_name="customer_name",hover_data=["city","area","margin_pct","aov"],
           title="Customer Revenue vs Profit").show()

rfm_summary = rfm.groupby("segment",as_index=False).agg(
    customers=("customer_id","nunique"),revenue=("monetary","sum"),
    avg_recency=("recency_days","mean"),avg_frequency=("frequency","mean")
)
display(rfm_summary)
px.bar(rfm_summary.sort_values("revenue"),x="revenue",y="segment",orientation="h",
       text_auto=".2s",title="Revenue by RFM-Style Segment").show()

,segment,customers,revenue,avg_recency,avg_frequency
0,At Risk,20,"936,167.89",22.45,25.35
1,High Value,18,"1,665,406.60",4.83,37.39
2,Low Value,4,"186,436.46",3.75,25.00
3,Loyal / Regular,38,"2,673,557.19",8.71,30.95
4,Regular,20,"1,115,042.72",8.85,27.20


# 8. Statistical Analysis

Tests are selected based on the structure of the data. Results indicate association/difference, not causation.

In [ ]:
def spearman_result(x,y,label):
    d=pd.concat([pd.Series(x),pd.Series(y)],axis=1).dropna()
    r,p=stats.spearmanr(d.iloc[:,0],d.iloc[:,1])
    return {"analysis":label,"n":len(d),"spearman_r":r,"p_value":p,"significant_5pct":p<.05}

corr = pd.DataFrame([
    spearman_result(sales.discount_pct,sales.profit_margin_pct,"Discount % vs Profit Margin %"),
    spearman_result(deliveries.distance_km,deliveries.delivery_cost,"Distance vs Delivery Cost"),
    spearman_result(deliveries.distance_km,deliveries.delivery_delay_days,"Distance vs Delivery Delay")
])
display(corr)

,analysis,n,spearman_r,p_value,significant_5pct
0,Discount % vs Profit Margin %,3000,-0.69,0.00,True
1,Distance vs Delivery Cost,2200,0.95,0.00,True
2,Distance vs Delivery Delay,2200,-0.02,0.31,False


## Supplier Performance Difference Test

Use Kruskal-Wallis because delivery-delay distributions may be non-normal.

- H0: supplier delay distributions are equal.
- H1: at least one supplier differs.

In [ ]:
supplier_groups = [g.supplier_delay_days.dropna().values for _,g in purchases.groupby("supplier_id") if len(g)>=2]
if len(supplier_groups)>=2:
    h,p = stats.kruskal(*supplier_groups)
    display(pd.DataFrame([{"test":"Kruskal-Wallis","H_statistic":h,"p_value":p,"significant_5pct":p<.05}]))
else:
    print("Insufficient groups for Kruskal-Wallis.")

,test,H_statistic,p_value,significant_5pct
0,Kruskal-Wallis,18.49,0.19,False


## Category Profit-Margin Difference Test

Use a normality diagnostic on capped samples. If the normality diagnostic is not supported across groups, use Kruskal-Wallis rather than forcing ANOVA.

In [ ]:
cat_data = sales.merge(products[["product_id","category"]],on="product_id")
groups = [g.profit_margin_pct.dropna().values for _,g in cat_data.groupby("category") if len(g)>1]

rng=np.random.default_rng(42)
normality_p=[]
for arr in groups:
    sample=rng.choice(arr,size=min(500,len(arr)),replace=False)
    normality_p.append(stats.shapiro(sample).pvalue if len(sample)>=3 else 0)

normality_supported = all(p>.05 for p in normality_p)

if normality_supported:
    statistic,p_value=stats.f_oneway(*groups)
    test_name="One-way ANOVA"
else:
    statistic,p_value=stats.kruskal(*groups)
    test_name="Kruskal-Wallis"

display(pd.DataFrame([{
    "test":test_name,"statistic":statistic,"p_value":p_value,
    "significant_5pct":p_value<.05,
    "normality_diagnostic_supported":normality_supported
}]))

,test,statistic,p_value,significant_5pct,normality_diagnostic_supported
0,Kruskal-Wallis,485.33,0.00,True,False


# 9. Visualization Quality Review

All analytical graphs use **Plotly Express**.

Before keeping a graph, ask:
1. What business question does it answer?
2. What decision could it support?
3. Is the chart type appropriate?
4. Is it clearer than a table?
5. Does it avoid implying causation?

Do not add charts merely to increase the chart count.

In [ ]:
# Warehouse scorecard
ws = sales.groupby("warehouse_id",as_index=False).agg(revenue=("sales_amount","sum"),profit=("profit","sum"))
wi = inventory[inventory.date.eq(latest_inventory_date)].groupby("warehouse_id",as_index=False).agg(
    inventory_value=("inventory_value","sum"),stockout_rate=("stockout_flag","mean"))
wd = deliveries.groupby("warehouse_id",as_index=False).agg(
    deliveries=("delivery_id","nunique"),
    on_time_rate=("delivery_delay_days",lambda x:(x<=0).mean()),
    avg_delay_days=("delivery_delay_days","mean"))

warehouse_scorecard = warehouses.merge(ws,on="warehouse_id",how="left").merge(wi,on="warehouse_id",how="left").merge(wd,on="warehouse_id",how="left")
warehouse_scorecard["stockout_rate_pct"]=warehouse_scorecard.stockout_rate*100
warehouse_scorecard["on_time_rate_pct"]=warehouse_scorecard.on_time_rate*100
display(warehouse_scorecard)

px.bar(warehouse_scorecard.sort_values("revenue"),x="revenue",y="warehouse_name",
       orientation="h",color="city",text_auto=".2s",title="Warehouse Revenue Performance").show()

,warehouse_id,warehouse_name,area,city,capacity_units,revenue,profit,inventory_value,stockout_rate,deliveries,on_time_rate,avg_delay_days,stockout_rate_pct,on_time_rate_pct
0,W001,East Delhi Warehouse,Gazipur,Delhi,15000,"2,789,916.33","458,025.33",1446955,0.10,968,0.48,0.71,10.00,47.73
1,W002,Noida Distribution Hub,Sector 62,Noida,12000,"1,690,801.33","273,714.33",1280509,0.03,549,0.46,0.72,3.33,45.90
2,W003,Ghaziabad Distribution Hub,Sahibabad,Ghaziabad,10000,"1,456,213.39","234,252.39",1062007,0.13,466,0.45,0.70,13.33,45.49
3,W004,Faridabad Distribution Hub,NIT,Faridabad,8000,"639,679.81","105,249.81",1328490,0.10,217,0.45,0.79,10.00,44.70


# 10. Insights & Findings

Generate quantified findings from the actual analysis. Do not write generic claims such as "sales increased" without evidence.

Use the format:

**Finding → Evidence → Business Meaning → Possible Action**

In [ ]:
top_category = category.sort_values("revenue",ascending=False).iloc[0]
top_profit_category = category.sort_values("profit",ascending=False).iloc[0]
top_product = product_summary.sort_values("revenue",ascending=False).iloc[0]
top_profit_product = product_summary.sort_values("profit",ascending=False).iloc[0]
top_stockout = stockouts.sort_values("stockout_rate_pct",ascending=False).iloc[0]
low_turn = inventory_metrics.dropna(subset=["inventory_turnover"]).sort_values("inventory_turnover").iloc[0]
best_supplier = supplier_summary.sort_values("on_time_rate_pct",ascending=False).iloc[0]
worst_supplier = supplier_summary.sort_values("on_time_rate_pct").iloc[0]
top_customer = customer_summary.sort_values("revenue",ascending=False).iloc[0]

insights = [
    f"**Revenue baseline:** ₹{total_revenue:,.0f} revenue across {total_orders:,} orders produced ₹{total_profit:,.0f} profit and a {profit_margin:.2f}% margin.",
    f"**Largest revenue category:** {top_category['category']} generated ₹{top_category['revenue']:,.0f}, or {top_category['revenue']/total_revenue*100:.1f}% of revenue.",
    f"**Largest profit category:** {top_profit_category['category']} generated ₹{top_profit_category['profit']:,.0f} profit.",
    f"**Top revenue SKU:** {top_product['product_name']} generated ₹{top_product['revenue']:,.0f}.",
    f"**Top profit SKU:** {top_profit_product['product_name']} generated ₹{top_profit_product['profit']:,.0f} profit.",
    f"**Inventory risk:** overall stockout observation rate is {stockout_rate:.2f}%; {top_stockout['product_name']} has the highest observed product stockout rate at {top_stockout['stockout_rate_pct']:.1f}%.",
    f"**Turnover risk:** {low_turn['product_name']} has the lowest valid calculated inventory turnover.",
    f"**Supplier spread:** best supplier on-time rate is {best_supplier['supplier_name']} at {best_supplier['on_time_rate_pct']:.1f}%, versus {worst_supplier['supplier_name']} at {worst_supplier['on_time_rate_pct']:.1f}%.",
    f"**Delivery:** overall delivery on-time rate is {delivery_on_time_rate:.1f}%.",
    f"**Customer concentration:** top revenue customer is {top_customer['customer_name']} with ₹{top_customer['revenue']:,.0f} revenue."
]
display(Markdown("\n".join([f"{i+1}. {x}" for i,x in enumerate(insights)])))

1. **Revenue baseline:** ₹6,576,611 revenue across 3,000 orders produced ₹1,071,242 profit and a 16.29% margin.
2. **Largest revenue category:** Staples generated ₹1,692,074, or 25.7% of revenue.
3. **Largest profit category:** Staples generated ₹243,564 profit.
4. **Top revenue SKU:** India Gate Rice 5kg generated ₹624,738.
5. **Top profit SKU:** India Gate Rice 5kg generated ₹83,658 profit.
6. **Inventory risk:** overall stockout observation rate is 14.65%; Pepsi 750ml has the highest observed product stockout rate at 41.7%.
7. **Turnover risk:** Coca-Cola 750ml has the lowest valid calculated inventory turnover.
8. **Supplier spread:** best supplier on-time rate is Capital Distributors at 84.3%, versus Shree Distributors at 64.9%.
9. **Delivery:** overall delivery on-time rate is 46.5%.
10. **Customer concentration:** top revenue customer is Kondli General Store 026 with ₹118,565 revenue.

## Data-Quality Findings

The dataset contains important limitations that should be disclosed in an interview.

In [ ]:
display(Markdown(
    f'''
### Inventory-sales reconciliation
Only **{int(recon_summary['exact_matches']):,} of {int(recon_summary['comparison_groups']):,}** month-product-warehouse groups reconcile exactly.
The total difference is **{recon_summary['total_difference']:,.0f} units**.

### Inventory stock equation
**{int(stock_check['mismatches']):,}** inventory rows do not satisfy the simple opening + received − sold − damaged = closing equation.

### Interpretation
These are data-quality limitations. The analysis keeps the original measures and does not silently "fix" them.
'''
))


### Inventory-sales reconciliation
Only **16 of 1,440** month-product-warehouse groups reconcile exactly.
The total difference is **-53,111 units**.

### Inventory stock equation
**52** inventory rows do not satisfy the simple opening + received − sold − damaged = closing equation.

### Interpretation
These are data-quality limitations. The analysis keeps the original measures and does not silently "fix" them.


# 11. Recommendations / Business Actions

### Quick Wins
- Review high-stockout SKUs and replenishment rules.
- Review slow-moving SKUs before additional purchasing.
- Introduce supplier scorecards.
- Review high-discount transactions with weak margins.
- Monitor warehouses with weaker delivery performance.

### Medium Term
- Apply ABC-based inventory policies.
- Calculate SKU-level reorder points and safety stock.
- Build supplier and delivery scorecards.
- Use customer segmentation for account prioritization.

### Long Term
- Integrate sales, inventory and procurement systems.
- Collect more history and build demand forecasting.
- Automate replenishment alerts.
- Add supplier risk and route optimization models.

Recommendations should be refined against validated operational data; do not claim financial savings unless they can be calculated.

In [ ]:
recommendations = pd.DataFrame([
["High-stockout SKU review","Stockout-rate analysis","High","Low","High"],
["Slow-moving SKU review","Low inventory turnover","High","Low","High"],
["Supplier scorecard","Supplier on-time-rate differences","Medium-High","Medium","High"],
["Discount/margin review","Discount-margin association","Medium","Low","Medium-High"],
["Warehouse delivery monitoring","Delivery on-time performance","High","Medium","High"],
["ABC inventory policy","Revenue concentration by SKU","High","Medium","High"],
["Demand forecasting","Only ~1 year of history","High","High","Medium"]
],columns=["Recommendation","Evidence","Expected Impact","Effort","Priority"])
display(recommendations)

,Recommendation,Evidence,Expected Impact,Effort,Priority
0,High-stockout SKU review,Stockout-rate analysis,High,Low,High
1,Slow-moving SKU review,Low inventory turnover,High,Low,High
2,Supplier scorecard,Supplier on-time-rate differences,Medium-High,Medium,High
3,Discount/margin review,Discount-margin association,Medium,Low,Medium-High
4,Warehouse delivery monitoring,Delivery on-time performance,High,Medium,High
5,ABC inventory policy,Revenue concentration by SKU,High,Medium,High
6,Demand forecasting,Only ~1 year of history,High,High,Medium


# 12. Conclusion

This project connects sales, inventory, procurement, customer and delivery data into one supply-chain intelligence workflow.

The analysis should answer:
- What drives revenue and profit?
- Where are the biggest inventory risks?
- Which suppliers require monitoring?
- Where are delivery bottlenecks?
- Which customers matter most?
- What data-quality issues limit operational confidence?

The key principle is:

> **Use the dashboard to identify the problem; use analysis to explain it; use operational action to solve it.**

# 13. Limitations & Next Steps

## Limitations
1. Inventory `sold_quantity` does not perfectly reconcile with sales transactions.
2. Inventory is periodic snapshot data rather than a continuous daily ledger.
3. Approximately one year of history limits seasonal forecasting.
4. Customer attributes are limited to supplied business fields.
5. Correlation does not prove causation.
6. If the dataset is simulated, findings demonstrate methodology rather than real company performance.
7. Delivery metrics apply only to records represented in `fact_delivery`.

## Next Steps
- Collect 2–3+ years of history.
- Reconcile inventory movements with sales/purchases.
- Add daily stock movements and adjustments.
- Build demand forecasting.
- Optimize safety stock/reorder points.
- Add supplier risk scoring.
- Add route/geographic optimization.
- Connect the workflow to SQL.
- Automate Power BI refreshes.

# Key Takeaways for Presentation

Use **Finding → Evidence → Business Impact**.

1. Commercial performance establishes the revenue/profit baseline.
2. ABC analysis identifies economically important SKUs.
3. Stockout and turnover analysis highlights inventory risks.
4. Revenue leaders and profit leaders should be compared separately.
5. Supplier on-time performance can support procurement scorecards.
6. Delivery reliability can reveal service-level bottlenecks.
7. RFM-style segmentation helps prioritize customer accounts.
8. Discount percentage should be evaluated against margin, not absolute discount.
9. Data-quality reconciliation is itself a business finding.
10. Better historical and operational data would enable forecasting and optimization.

# Interview Preparation

Be ready to explain the project in this order:

**Business Problem → Data → Cleaning → Analysis → Key Finding → Business Action**

Questions to prepare:

### Business
1. Why did you choose this project?
2. What was the business problem?
3. Which KPI matters most and why?
4. What would you present to management first?

### Python/Pandas
5. How did you validate multiple CSVs?
6. Why did you create reusable functions?
7. How did you calculate MoM growth?
8. How did you handle division by zero?
9. How did you reconcile inventory and sales?

### SQL
10. How would you calculate monthly revenue?
11. How would you rank products using window functions?
12. How would you calculate supplier on-time rate?
13. How would you identify slow-moving SKUs?

### Statistics
14. Why Spearman correlation?
15. What is correlation vs causation?
16. Why Kruskal-Wallis instead of ANOVA?
17. What does p < 0.05 mean?

### Power BI
18. How would you design the star schema?
19. What DAX measures would you create?
20. How would you implement time intelligence?
21. How would you design the executive dashboard?

### Data Quality / Scenarios
22. What did you find in inventory-sales reconciliation?
23. Why can't monthly inventory snapshots simply be summed?
24. What would you investigate if a warehouse had both high inventory and high stockouts?
25. What additional data would you request before automated replenishment?

# Final Data Analyst Quality Checklist

| Check | Status |
|---|---|
| Business problem defined | Completed |
| Objectives/questions defined | Completed |
| All supplied CSVs loaded | Completed |
| Row counts/date ranges verified | Completed |
| Primary keys validated | Completed |
| Foreign keys validated | Completed |
| Missing/duplicate/negative checks | Completed |
| Sales/profit calculations validated | Completed |
| Inventory-sales reconciliation | Completed |
| Inventory stock equation | Completed |
| Feature engineering | Completed |
| ABC analysis | Completed |
| RFM-style segmentation | Completed |
| EDA | Completed |
| Statistical analysis | Completed |
| Plotly Express visualizations | Completed |
| Quantified insights | Completed |
| Recommendations | Completed |
| Limitations | Completed |
| Interview preparation | Completed |
| Correlation vs causation addressed | Completed |

## Final Principle

**Think like a Data Analyst, not a chart generator.**

**Data → Business Question → Analysis → Evidence → Insight → Business Action**